# Is there a fast way to solve any Sudoku?

Checking a solution is fast. Finding one is search. That asymmetry is the whole
of NP in one puzzle.

In [1]:
puzzle = [
    "53..7....", "6..195...", ".98....6.",
    "8...6...3", "4..8.3..1", "7...2...6",
    ".6....28.", "...419..5", "....8..79",
]
grid = [[0 if ch == "." else int(ch) for ch in row] for row in puzzle]

In [2]:
def valid(grid):
    """The verifier: every row, column and box holds 1..9 exactly once. Polynomial."""
    full = set(range(1, 10))
    rows = all(set(r) == full for r in grid)
    cols = all({grid[r][c] for r in range(9)} == full for c in range(9))
    boxes = all({grid[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3)} == full
                for br in (0, 3, 6) for bc in (0, 3, 6))
    return rows and cols and boxes

def solve(grid, stats):
    """The solver: backtracking search. Exponential in the worst case."""
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                for v in range(1, 10):
                    if all(v != grid[r][k] for k in range(9)) and all(v != grid[k][c] for k in range(9)) \
                       and all(v != grid[br][bc] for br in range(r // 3 * 3, r // 3 * 3 + 3) for bc in range(c // 3 * 3, c // 3 * 3 + 3)):
                        grid[r][c] = v
                        stats["placements"] += 1
                        if solve(grid, stats):
                            return True
                        grid[r][c] = 0          # backtrack
                return False
    return True

In [3]:
import time
stats = {"placements": 0}
t0 = time.perf_counter(); solve(grid, stats); t_solve = time.perf_counter() - t0
t0 = time.perf_counter(); ok = valid(grid);   t_check = time.perf_counter() - t0
for row in grid: print(" ".join(map(str, row)))
print(f"\nsolved in {t_solve:.3f} s with {stats['placements']} placements;  verified in {t_check * 1e6:.0f} µs -> {ok}")

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9

solved in 0.028 s with 4208 placements;  verified in 47 µs -> True


Checking took microseconds; finding took thousands of tries. For $9 \times 9$
that is fine. Generalized Sudoku ($n^2 \times n^2$) is NP-complete, so a
polynomial-time solver for it would give one for every problem in NP.

## Verify fast, find slow: the definition of NP

`valid` is a polynomial-time *verifier*; that alone puts Sudoku in NP. The
reduction from other NP problems is what makes it NP-*complete*.

**What we used:** P, NP, verifiers and NP-completeness (session 9, sections 7 and 9).